# 04 -- Feature Build

> **Data mode: SYNTHETIC FIXTURE DATA.** Every filing/price/universe/sector record in this notebook comes from `_fixtures.py`, hand-authored for demonstration only. **Nothing computed in this notebook is a genuine historical result, and no cell here may be cited as evidence of reproducing `report_current.html`.** See `docs/reproducibility_findings.md` for the actual reproducibility status.

Runs Stage 3's real feature pipeline (`build_production_features`) over the synthetic bundle -- no formula in this notebook; every feature value comes from `atlas_quant.strategies.filing_momentum_ml.feature_pipeline`, unchanged.

## Shared strategy cohort vs. issuer fiscal history

A correction was made after Stage 11's real-data acquisition surfaced an implementation bug: `build_feature_observation` originally required a filing's own fiscal `quarter_end` to *exactly equal* the shared calendar cohort's date as a condition for the row to exist at all. Synthetic test fixtures are always calendar-aligned, so this was invisible until real data -- where most real issuers use 52/53-week or otherwise offset fiscal years (confirmed: only 78.4% of 96,852 real acquired filing rows are exactly calendar-aligned; AAPL's fiscal quarters end on the last Saturday of the month, a few days off calendar quarter-end; WMT's fiscal year ends January 31, a full month offset).

The recovered behavior (reverse-engineered from `report_current.html`'s own `get_available_as_of` -- "capped at the most recent 8 filed quarters", an *ordinal* rule -- and the legacy `RollingMLScorer`/`data_sec.py` implementation): every ticker produces **one candidate row per shared calendar cohort**, built from whichever fiscal history is most recently knowable as of that cohort's own buy timestamp. Exact calendar alignment is used **only** to refine entry-timing precision (`filed_at + 1 trading day`, capped at the cohort's buy timestamp) -- never as a requirement for the row to exist. Absent an exact match, the feature timestamp falls back to the cohort's own buy timestamp directly -- never nearest-date matching, never a rejection.

`FeatureObservation` now carries both concepts explicitly, never conflated: `quarter_end`/`fiscal_period` (the issuer's own actual fiscal quarter) and `strategy_cohort_end`/`cohort_buy_timestamp` (the shared calendar cohort used for global labeling, rolling training windows, portfolio entry/exit, and benchmark comparison). See `docs/reproducibility_findings.md` for the full investigation, real-data recovery counts, and why a "nearest calendar quarter" mapping was considered and rejected.

In [ ]:
import atlas_quant
print("atlas_quant path:", atlas_quant.__file__)

import sys
sys.path.insert(0, ".")  # this notebook's own directory (for _fixtures.py) -- not a legacy path hack
import _fixtures


In [ ]:
from datetime import date, datetime
from atlas_quant.strategies.filing_momentum_ml.config import FeatureCacheIdentity, FilingMomentumMLConfig
from atlas_quant.strategies.filing_momentum_ml.production.feature_label_build import build_production_features
from atlas_quant.strategies.filing_momentum_ml.sector_encoding import SectorEncoder

bundle = _fixtures.build_synthetic_bundle()
calendar = _fixtures.build_trading_calendar(bundle)
periods = _fixtures.build_periods()
config = FilingMomentumMLConfig()
# Each shared cohort carries its own cohort_buy_timestamp (== that
# period's own entry_timestamp) as its point-in-time cutoff.
targets = [(i, p.quarter_end, p.entry_timestamp) for p in periods for i in bundle.universe]

cache_identity = FeatureCacheIdentity(
    strategy_id=config.strategy_id, strategy_version="notebook", feature_schema_version="2",
    fcf_mode=config.fcf_mode, train_years=config.ml_train_years, min_train_quarters=config.min_train_quarters,
    model_config_identity=config.identity(), universe_id="synthetic-notebook-universe",
    data_cutoff=date(2023, 6, 1), created_at=datetime(2023, 6, 1),
)
result = build_production_features(
    config=config, calendar=calendar, sector_encoder=SectorEncoder(), targets=targets,
    filings_by_instrument=bundle.filings_by_instrument, prices_by_instrument=bundle.prices_by_instrument,
    sector_by_instrument=bundle.sector_by_instrument,
    cache_identity=cache_identity, cache_root=None,  # never write to a real cache path from a notebook
)
print("blocked:", result.blocked)
for obs in result.feature_pipeline_result.observations:
    print(
        obs.instrument_id.symbol, "fiscal_quarter_end=", obs.quarter_end,
        "strategy_cohort_end=", obs.strategy_cohort_end,
        "missing_features:", obs.missing_features,
    )
    print("  ", {k: round(v, 4) for k, v in obs.features.items()})


## Findings

- Both synthetic instruments produce a `FeatureObservation` for the target quarter, with a handful of missing features (the synthetic price series is too short/synthetic for every momentum/volatility window to be fully populated).

## Limitations

- These feature *values* are meaningless (derived from a smooth, synthetic price/revenue series) -- only the *mechanism* (point-in-time selection, timing resolution, missing-data handling) is being demonstrated.